# QOS on Real Quantum Hardware (IBM)
**Tommaso R. Marena (2026)**

Runs the Quantum Oracle Sketching (QOS) Boolean-phase oracle circuit on real IBM quantum hardware
via **Qiskit Runtime** and compares results against ideal statevector simulation.

**What this notebook does:**
1. Builds small QOS oracle circuits (n=3–5 qubits) from the phase-diagonal returned by `q_oracle_sketch_boolean`
2. Transpiles to IBM native gate set (ECR/CZ basis) with optimization level 3
3. Runs on the least-busy IBM QPU (or ibm_brisbane/ibm_kyoto if specified)
4. Applies **Zero-Noise Extrapolation (ZNE)** via Qiskit IBM Runtime Estimator V2
5. Computes fidelity vs. ideal statevector and total variation distance
6. Saves results to Google Drive for inclusion in paper Section 6 / Appendix B

> **Cost:** Each n=5 circuit uses ~200 shots x 3 noise factors = 600 shots total.
> At IBM free tier (10 min/month) this runs in under 3 minutes.
> Results are reproducible: circuits are fixed by `SEED`.

## 0. Setup & Authentication

In [ ]:
# Install QOS + Qiskit Runtime
import subprocess, sys
pkgs = [
    'quantum-oracle-sketching[dev,noise,kernel] @ git+https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git',
    'qiskit>=1.1',
    'qiskit-ibm-runtime>=0.23',
    'qiskit-aer>=0.14',
    'matplotlib',
    'numpy',
]
for pkg in pkgs:
    subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True)
print('All packages installed.')

In [ ]:
import os
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/qos_hardware'
else:
    OUTPUT_DIR = '/content/qos_hardware'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ============================================================
# IBM QUANTUM AUTHENTICATION
# Get your token at: https://quantum.ibm.com/account
# Paste it below OR set the IBMQ_TOKEN environment variable.
# ============================================================
from qiskit_ibm_runtime import QiskitRuntimeService

IBM_TOKEN = os.environ.get('IBMQ_TOKEN', '')  # set env var or paste token here
if not IBM_TOKEN:
    from getpass import getpass
    IBM_TOKEN = getpass('Paste your IBM Quantum token: ')

QiskitRuntimeService.save_account(channel='ibm_quantum', token=IBM_TOKEN, overwrite=True)
service = QiskitRuntimeService(channel='ibm_quantum')
print('Authenticated. Available backends:')
for b in service.backends(simulator=False, operational=True):
    print(f'  {b.name:30s}  qubits={b.num_qubits}  pending={b.status().pending_jobs}')

In [ ]:
import numpy as np, jax, jax.numpy as jnp

# ── Hardware config ─────────────────────────────────────────
BACKEND_NAME   = 'least_busy'   # 'least_busy' | 'ibm_brisbane' | 'ibm_kyoto'
SHOTS          = 4096           # shots per circuit per noise factor
USE_ZNE        = True           # Zero-Noise Extrapolation
ZNE_FACTORS    = [1, 2, 3]      # noise scale factors for ZNE
OPT_LEVEL      = 3              # Qiskit transpiler optimisation level
SEED           = 42             # reproducibility

# ── Circuit sizes to benchmark ──────────────────────────────
# n=3 (8 inputs): fits any IBM 5-qubit device (ibmq_manila etc)
# n=4 (16 inputs): needs 6+ qubits
# n=5 (32 inputs): needs 7+ qubits, main paper result
N_QUBITS_LIST  = [3, 4, 5]
M_SAMPLES      = 1024           # QOS sample budget M (matching Zhao et al.)

# ── Select backend ──────────────────────────────────────────
if BACKEND_NAME == 'least_busy':
    backend = service.least_busy(
        simulator=False, operational=True,
        filters=lambda b: b.num_qubits >= max(N_QUBITS_LIST) + 2
    )
else:
    backend = service.backend(BACKEND_NAME)
print(f'Using backend: {backend.name}  ({backend.num_qubits} qubits)')
print(f'Pending jobs:  {backend.status().pending_jobs}')


## 1. Build QOS Oracle Circuits

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import PhaseGate
from qos.core.oracle_sketch import q_oracle_sketch_boolean

rng = np.random.default_rng(SEED)

def build_qos_circuit(n_qubits: int, M: int, seed: int = 42) -> tuple:
    """
    Build a QOS Boolean phase-oracle circuit for a random n-bit truth table.

    Strategy:
    ---------
    1. Sample a random Boolean truth table f: {0,1}^n -> {0,1}
    2. Compute the QOS phase diagonal via q_oracle_sketch_boolean()
    3. Decompose the diagonal unitary into single-qubit phase gates +
       multi-controlled phase gates via Gray-code Walsh-Hadamard decomposition
    4. Prepend H^{⊗n} to create a uniform superposition input state
    5. Append H^{⊗n} + measurement to return to computational basis

    Returns: (circuit, truth_table, ideal_probs)
    """
    N = 2 ** n_qubits
    rng_local = np.random.default_rng(seed)
    truth_table = jnp.array(rng_local.integers(0, 2, size=N), dtype=jnp.float64)

    # QOS phase diagonal (complex array of length N)
    diag, _ = q_oracle_sketch_boolean(truth_table, M)
    phases = jnp.angle(diag)  # extract phase angles

    # ── Walsh-Hadamard decomposition of diagonal unitary ────
    # Any diagonal unitary D = diag(e^{i*phi_x}) can be written as
    # a sum over Pauli-Z tensor products. The coefficients are the
    # Walsh-Hadamard transform of the phase vector.
    phases_np = np.array(phases)
    # WHT: theta_k = (1/N) * sum_x phi_x * (-1)^{popcount(x & k)}
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    Hn = H.copy()
    for _ in range(n_qubits - 1):
        Hn = np.kron(Hn, H)
    wht_coeffs = (Hn @ phases_np) / N  # coefficients theta_k

    # ── Build circuit ────────────────────────────────────────
    qr = QuantumRegister(n_qubits, 'q')
    cr = ClassicalRegister(n_qubits, 'c')
    qc = QuantumCircuit(qr, cr)

    # Uniform superposition
    qc.h(qr)

    # Apply diagonal unitary via WHT decomposition
    # For each non-zero coefficient k, apply controlled-phase
    # The k-th coefficient corresponds to Z^{⊗popcount(k)} on qubits in support(k)
    EPS = 1e-8
    for k in range(N):
        theta = float(wht_coeffs[k])
        if abs(theta) < EPS:
            continue
        bits = [j for j in range(n_qubits) if (k >> j) & 1]
        if len(bits) == 0:
            # Global phase -- physically unobservable, skip
            continue
        elif len(bits) == 1:
            qc.p(2 * theta, qr[bits[0]])
        else:
            # Multi-controlled phase: decompose with CNOT ladder
            # CX chain to accumulate parity onto last qubit, then P, then uncompute
            ctrl_bits = bits[:-1]
            tgt       = bits[-1]
            for c in ctrl_bits:
                qc.cx(qr[c], qr[tgt])
            qc.p(2 * theta, qr[tgt])
            for c in reversed(ctrl_bits):
                qc.cx(qr[c], qr[tgt])

    # Inverse H to read out in computational basis
    qc.h(qr)
    qc.measure(qr, cr)

    # ── Ideal output probabilities (statevector) ─────────────
    # |psi_out> = H^n D H^n |0>^n
    # Since H^n |0>^n = uniform superposition, and we apply D then H^n,
    # ideal probs = |IFFT(diag)|^2 (up to normalisation)
    state = np.ones(N, dtype=complex) / np.sqrt(N)
    state = np.array(diag) * state
    state = Hn @ state
    ideal_probs = np.abs(state) ** 2
    ideal_probs /= ideal_probs.sum()

    return qc, np.array(truth_table), ideal_probs


# Build and display circuits
circuits = {}
ideal_probs_all = {}
truth_tables = {}

for n in N_QUBITS_LIST:
    qc, tt, ip = build_qos_circuit(n, M_SAMPLES, seed=SEED)
    circuits[n] = qc
    truth_tables[n] = tt
    ideal_probs_all[n] = ip
    print(f'n={n}: depth={qc.depth()}, gates={qc.count_ops()}, ideal_H(p)={-np.sum(ip*np.log2(ip+1e-15)):.3f} bits')


## 2. Transpile to Hardware Native Gate Set

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

transpiled = {}
for n in N_QUBITS_LIST:
    pm = generate_preset_pass_manager(
        optimization_level=OPT_LEVEL,
        backend=backend,
        seed_transpiler=SEED,
    )
    t_qc = pm.run(circuits[n])
    transpiled[n] = t_qc
    cx_count = t_qc.count_ops().get('cx', 0) + t_qc.count_ops().get('ecr', 0) + t_qc.count_ops().get('cz', 0)
    print(f'n={n}: transpiled_depth={t_qc.depth()}, 2Q_gates={cx_count}')


## 3. Noisy Simulation (Aer + IBM Noise Model)

In [ ]:
# Run on Aer with the real IBM noise model first — fast, free, validates the pipeline
# before spending IBM QPU minutes.
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeKyotoV2
from qiskit import transpile

fake_backend = FakeKyotoV2()
noise_model  = NoiseModel.from_backend(fake_backend)
aer_sim      = AerSimulator(noise_model=noise_model)

aer_results = {}
for n in N_QUBITS_LIST:
    qc_t = transpile(circuits[n], aer_sim, optimization_level=OPT_LEVEL, seed_transpiler=SEED)
    job  = aer_sim.run(qc_t, shots=SHOTS, seed_simulator=SEED)
    counts = job.result().get_counts()
    # Convert counts to probability vector
    N = 2 ** n
    probs = np.zeros(N)
    for bitstr, cnt in counts.items():
        idx = int(bitstr, 2)
        probs[idx] = cnt / SHOTS
    aer_results[n] = probs
    tvd = 0.5 * np.sum(np.abs(probs - ideal_probs_all[n]))
    print(f'n={n} Aer(noisy): TVD={tvd:.4f}')


## 4. Run on Real IBM QPU
> This cell submits jobs to real quantum hardware. Check your IBM Quantum account
> for job status at https://quantum.ibm.com/jobs
> 
> **Estimated QPU time:** ~1-3 min for all three circuit sizes.
> The job IDs are saved so you can reload results without resubmitting.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler, Batch
import json, time

job_ids_path = os.path.join(OUTPUT_DIR, 'hardware_job_ids.json')

# ── Submit jobs (or reload saved job IDs) ────────────────────
if os.path.exists(job_ids_path):
    with open(job_ids_path) as f:
        saved = json.load(f)
    print(f'Reloading saved job IDs: {saved}')
    jobs = {int(k): service.job(v) for k, v in saved.items()}
else:
    jobs = {}
    job_id_map = {}
    with Batch(backend=backend) as batch:
        sampler = Sampler(mode=batch)
        sampler.options.default_shots = SHOTS
        for n in N_QUBITS_LIST:
            pub = (transpiled[n],)
            job = sampler.run([pub])
            jobs[n] = job
            job_id_map[n] = job.job_id()
            print(f'n={n}: submitted job {job.job_id()}')
    with open(job_ids_path, 'w') as f:
        json.dump(job_id_map, f, indent=2)
    print(f'Job IDs saved to {job_ids_path}')

# ── Wait for completion ──────────────────────────────────────
print('Waiting for jobs to complete...')
hw_results_raw = {}
for n, job in jobs.items():
    t0 = time.time()
    result = job.result()
    elapsed = time.time() - t0
    pub_result = result[0]
    counts = pub_result.data.c.get_counts()
    N_states = 2 ** n
    probs = np.zeros(N_states)
    for bitstr, cnt in counts.items():
        idx = int(bitstr, 2)
        probs[idx] = cnt / SHOTS
    hw_results_raw[n] = probs
    tvd = 0.5 * np.sum(np.abs(probs - ideal_probs_all[n]))
    print(f'n={n}: TVD(raw)={tvd:.4f}  elapsed={elapsed:.1f}s')

## 5. Zero-Noise Extrapolation (ZNE)

In [ ]:
# ZNE via gate-folding: repeat each 2Q gate k times to scale noise by factor k.
# Then fit a polynomial to the noisy expectations and extrapolate to lambda=0.
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.quantum_info import SparsePauliOp

def fold_gates(qc, noise_factor: int):
    """Gate-fold: repeat each 2Q gate (noise_factor-1)/2 extra times (odd factors only)."""
    assert noise_factor % 2 == 1, 'ZNE noise factors must be odd integers'
    if noise_factor == 1:
        return qc
    from qiskit import QuantumCircuit
    new_qc = QuantumCircuit(*qc.qregs, *qc.cregs)
    for instr in qc.data:
        new_qc.append(instr)
        # Fold 2Q gates by appending inverse+forward pairs
        if instr.operation.num_qubits == 2:
            for _ in range((noise_factor - 1) // 2):
                new_qc.append(instr.operation.inverse(), instr.qubits, instr.clbits)
                new_qc.append(instr)
    return new_qc

if USE_ZNE and os.path.exists(job_ids_path):
    print('Running ZNE with noise factors:', ZNE_FACTORS)
    zne_jobs = {n: {} for n in N_QUBITS_LIST}
    zne_job_ids = {}

    with Batch(backend=backend) as batch:
        sampler = Sampler(mode=batch)
        sampler.options.default_shots = SHOTS
        for n in N_QUBITS_LIST:
            zne_job_ids[n] = {}
            for lam in ZNE_FACTORS:
                folded = fold_gates(transpiled[n], lam)
                pub    = (folded,)
                job    = sampler.run([pub])
                zne_jobs[n][lam]    = job
                zne_job_ids[n][lam] = job.job_id()
                print(f'  n={n} lam={lam}: submitted {job.job_id()}')

    with open(os.path.join(OUTPUT_DIR,'zne_job_ids.json'),'w') as f:
        json.dump(zne_job_ids, f, indent=2)

    # Collect ZNE results and extrapolate
    hw_results_zne = {}
    for n in N_QUBITS_LIST:
        N_states = 2 ** n
        noisy_probs = []
        for lam in ZNE_FACTORS:
            result = zne_jobs[n][lam].result()
            counts = result[0].data.c.get_counts()
            probs  = np.zeros(N_states)
            for bitstr, cnt in counts.items():
                probs[int(bitstr, 2)] = cnt / SHOTS
            noisy_probs.append(probs)
        # Polynomial ZNE extrapolation: fit degree-1 poly to (lambda, p) pairs
        lam_arr    = np.array(ZNE_FACTORS, dtype=float)
        noisy_arr  = np.stack(noisy_probs, axis=0)  # (n_factors, N_states)
        zne_probs  = np.zeros(N_states)
        for s in range(N_states):
            coeffs = np.polyfit(lam_arr, noisy_arr[:, s], deg=min(2, len(ZNE_FACTORS)-1))
            zne_probs[s] = np.polyval(coeffs, 0.0)  # extrapolate to lambda=0
        zne_probs = np.clip(zne_probs, 0, None)
        zne_probs /= zne_probs.sum()
        hw_results_zne[n] = zne_probs
        tvd_raw = 0.5 * np.sum(np.abs(hw_results_raw[n] - ideal_probs_all[n]))
        tvd_zne = 0.5 * np.sum(np.abs(zne_probs - ideal_probs_all[n]))
        print(f'n={n}: TVD_raw={tvd_raw:.4f}  TVD_ZNE={tvd_zne:.4f}  improvement={tvd_raw-tvd_zne:+.4f}')
else:
    print('ZNE skipped (USE_ZNE=False or jobs not yet submitted).')
    hw_results_zne = hw_results_raw


## 6. Fidelity & Total Variation Distance

In [ ]:
import json

def classical_fidelity(p: np.ndarray, q: np.ndarray) -> float:
    """Bhattacharyya / classical fidelity F = (sum sqrt(p_i * q_i))^2"""
    return float(np.sum(np.sqrt(np.clip(p, 0, None) * np.clip(q, 0, None))) ** 2)

def tvd(p: np.ndarray, q: np.ndarray) -> float:
    return float(0.5 * np.sum(np.abs(p - q)))

print(f"{'n':>4} {'TVD_raw':>10} {'TVD_ZNE':>10} {'F_raw':>10} {'F_ZNE':>10} {'TVD_Aer':>10}")
print('-' * 55)
summary = {}
for n in N_QUBITS_LIST:
    ip = ideal_probs_all[n]
    raw_tvd  = tvd(hw_results_raw[n], ip)
    zne_tvd  = tvd(hw_results_zne[n], ip)
    raw_fid  = classical_fidelity(hw_results_raw[n], ip)
    zne_fid  = classical_fidelity(hw_results_zne[n], ip)
    aer_tvd  = tvd(aer_results[n], ip) if n in aer_results else float('nan')
    print(f'{n:>4} {raw_tvd:>10.4f} {zne_tvd:>10.4f} {raw_fid:>10.4f} {zne_fid:>10.4f} {aer_tvd:>10.4f}')
    summary[n] = {'n_qubits': n, 'M': M_SAMPLES, 'shots': SHOTS,
                  'tvd_raw': raw_tvd, 'tvd_zne': zne_tvd,
                  'fidelity_raw': raw_fid, 'fidelity_zne': zne_fid,
                  'tvd_aer_noisy': aer_tvd,
                  'backend': backend.name, 'zne_factors': ZNE_FACTORS}

out_path = os.path.join(OUTPUT_DIR, 'hardware_results.json')
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {out_path}')

## 7. Plots

In [ ]:
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10, 'font.family': 'serif'})

fig, axes = plt.subplots(1, len(N_QUBITS_LIST), figsize=(5 * len(N_QUBITS_LIST), 4))
if len(N_QUBITS_LIST) == 1: axes = [axes]

for ax, n in zip(axes, N_QUBITS_LIST):
    N_states = 2 ** n
    x = np.arange(N_states)
    ax.bar(x - 0.3, ideal_probs_all[n],    0.25, label='Ideal (statevec)', color='steelblue', alpha=0.85)
    ax.bar(x,        hw_results_raw[n],     0.25, label='QPU (raw)',        color='tomato',    alpha=0.85)
    ax.bar(x + 0.3,  hw_results_zne[n],     0.25, label='QPU (ZNE)',        color='seagreen',  alpha=0.85)
    tvd_zne = tvd(hw_results_zne[n], ideal_probs_all[n])
    ax.set_title(f'n={n} qubits\nTVD(ZNE)={tvd_zne:.3f}')
    ax.set_xlabel('Basis state $|x\rangle$')
    ax.set_ylabel('Probability')
    ax.legend(fontsize=8)
    ax.set_ylim(0, None)

plt.suptitle(f'QOS Boolean Oracle — IBM {backend.name}\n(M={M_SAMPLES} samples, {SHOTS} shots/circuit)',
             fontsize=11, y=1.01)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'hardware_oracle_probs.pdf')
plt.savefig(plot_path, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

# ── TVD vs n_qubits ──────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(5, 3.5))
ns     = list(summary.keys())
tvd_r  = [summary[n]['tvd_raw']  for n in ns]
tvd_z  = [summary[n]['tvd_zne']  for n in ns]
tvd_a  = [summary[n]['tvd_aer_noisy'] for n in ns]
ax2.plot(ns, tvd_r, 'o--', color='tomato',   label='QPU raw')
ax2.plot(ns, tvd_z, 's-',  color='seagreen',  label='QPU ZNE')
ax2.plot(ns, tvd_a, '^:',  color='goldenrod', label='Aer (noise model)')
ax2.set_xlabel('Number of qubits $n$')
ax2.set_ylabel('Total Variation Distance')
ax2.set_title('QOS Oracle Circuit Fidelity vs Circuit Size')
ax2.legend()
ax2.set_xticks(ns)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
tvd_path = os.path.join(OUTPUT_DIR, 'hardware_tvd_vs_n.pdf')
plt.savefig(tvd_path, bbox_inches='tight')
plt.show()
print(f'Saved: {tvd_path}')

## 8. Paper-Ready Result Summary
Run this cell last to generate the LaTeX table snippet for Appendix B.

In [ ]:
print('='*60)
print('HARDWARE RESULTS SUMMARY')
print('='*60)
print(f'Backend:      {backend.name}')
print(f'Shots/circuit: {SHOTS}')
print(f'ZNE factors:  {ZNE_FACTORS}')
print(f'M (QOS):      {M_SAMPLES}')
print()
print(f"{'n':>4} {'Fidelity(ZNE)':>14} {'TVD(ZNE)':>10} {'TVD(raw)':>10}")
print('-'*42)
for n in N_QUBITS_LIST:
    s = summary[n]
    print(f"{n:>4} {s['fidelity_zne']:>14.4f} {s['tvd_zne']:>10.4f} {s['tvd_raw']:>10.4f}")
print()
print('--- LaTeX table snippet (Appendix B) ---')
print(r'\begin{tabular}{cccc}')
print(r'\toprule')
print(r'$n$ & Fidelity (ZNE) & TVD (ZNE) & TVD (raw) \\')
print(r'\midrule')
for n in N_QUBITS_LIST:
    s = summary[n]
    print(f"{n} & {s['fidelity_zne']:.4f} & {s['tvd_zne']:.4f} & {s['tvd_raw']:.4f} \\\\")
print(r'\bottomrule')
print(r'\end{tabular}')
